<a href="https://colab.research.google.com/github/chaeyun050821-sketch/20241263/blob/main/Assignment_08_Hashing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

20241263 이채윤
## Assignment 08 - HASHING

---
## 문제 1. 충돌 해결 두 가지 방법의 장단점 비교

해싱에서 충돌이 발생할 경우 크게 **개방 주소법(Open Addressing)** 과 **체이닝(Chaining, 분리 연결법)** 두 가지 방법으로 해결할 수 있다.

### 1) 개방 주소법 (Open Addressing)

충돌 발생 시 해시 테이블 **내부의 다른 빈 슬롯**을 탐색하여 데이터를 저장하는 방식이다.  
대표적인 탐색 방법으로 선형 탐색(Linear Probing), 이차 탐색(Quadratic Probing), 더블 해싱(Double Hashing)이 있다.

| 구분 | 내용 |
|------|------|
| **장점** | 포인터를 사용하지 않으므로 추가 메모리 없이 테이블 하나만 사용 → **메모리 효율** 우수 |
| **장점** | 데이터가 테이블 배열 안에 연속적으로 저장되어 **캐시 성능(cache locality)** 이 좋음 |
| **장점** | 구현이 비교적 단순함 |
| **단점** | 테이블이 가득 찰수록 충돌이 급격히 증가하고 성능이 저하됨 (적재율 α < 0.7 권장) |
| **단점** | **군집화(Clustering)** 현상이 발생할 수 있음 (특히 선형 탐색) |
| **단점** | 삭제가 복잡함 — 단순히 삭제하면 탐색 체인이 끊겨 별도의 삭제 마커(tombstone)가 필요 |

---

### 2) 체이닝 (Chaining / Separate Chaining)

충돌 발생 시 같은 버킷에 해당하는 키들을 **연결 리스트(linked list)** 로 연결하여 저장하는 방식이다.

| 구분 | 내용 |
|------|------|
| **장점** | 테이블 크기보다 많은 키도 저장 가능 → **적재율(load factor)** 제한이 없음 |
| **장점** | 삽입과 삭제가 단순함 (연결 리스트 연산 사용) |
| **장점** | 군집화 문제가 발생하지 않음 |
| **단점** | 각 버킷마다 연결 리스트 포인터가 필요하므로 **추가 메모리** 소비 |
| **단점** | 포인터를 따라가는 과정에서 **캐시 성능** 이 개방 주소법보다 나쁨 |
| **단점** | 충돌이 많아지면 리스트가 길어져 최악의 경우 O(n) 탐색 시간 |

---

### 요약 비교표

| 비교 항목 | 개방 주소법 | 체이닝 |
|-----------|------------|--------|
| 메모리 사용 | 적음 (배열만 사용) | 많음 (포인터 추가) |
| 캐시 효율 | 높음 | 낮음 |
| 적재율 한계 | α < 1 (보통 0.7 이하 권장) | 제한 없음 |
| 삭제 복잡도 | 높음 (tombstone 필요) | 낮음 |
| 군집화 문제 | 있음 | 없음 |
| 구현 복잡도 | 비교적 단순 | 단순 |

---
## 문제 2-01. 해시 테이블 크기 1024, h(x) = x%1024의 결점

**결점 분석:**

1. **2의 거듭제곱 문제**: 테이블 크기 1024 = 2¹⁰이고, h(x) = x % 1024는 x의 **하위 10비트만** 사용한다.  
   → 키의 상위 비트 정보를 전혀 활용하지 못하므로 **고른 분포**를 보장하기 어렵다.

2. **편향된 분포**: 예를 들어 키 값들이 짝수나 특정 배수로만 이루어진 경우, 테이블의 절반 이상은 비어있고 나머지 절반에만 집중되는 심각한 **군집 현상**이 발생한다.

3. **소수(prime number)를 사용하지 않음**: 일반적으로 해시 테이블 크기는 소수로 설정해야 키가 고르게 분포된다.  
   2의 거듭제곱인 1024는 나머지 연산 시 특정 패턴이 반복될 가능성이 높다.

**개선 방법**: 테이블 크기를 1024에 가까운 소수, 예를 들어 **1021**로 변경하고 h(x) = x % 1021을 사용하면 보다 균일한 분포를 얻을 수 있다.

---
## 문제 2-02. h(x) = x * random(0,10) % m 이 해시 함수가 될 수 없는 이유

**이유:**

해시 함수는 반드시 **결정론적(deterministic)** 이어야 한다.  
즉, 동일한 입력 키 x에 대해 **항상 동일한 해시 값**을 반환해야 한다.

그런데 h(x) = x * random(0,10) % m 에서 `random(0,10)`은 호출할 때마다 **다른 임의의 실수**를 반환한다.  
따라서 같은 키 x를 넣어도 매번 다른 해시 값이 계산된다.

**문제점 예시:**
- 키 x = 5를 삽입할 때: h(5) = 5 * 3.7 % m = 18 % m → 슬롯 18에 저장
- 키 x = 5를 검색할 때: h(5) = 5 * 7.2 % m = 36 % m → 슬롯 36을 탐색 → **찾지 못함**

이처럼 **삽입 시 계산한 위치와 검색 시 계산한 위치가 달라져** 정상적인 검색이 불가능하다.  
따라서 이 함수는 해시 함수의 기본 요건을 위반하므로 해시 함수로 사용할 수 없다.

---
## 문제 2-04. 선형 탐색(Linear Probing)으로 키 삽입

In [1]:
def linear_probing_insert(table, key, size=13):
    """선형 탐색으로 해시 테이블에 키 삽입"""
    h = key % size  # 기본 해시 값
    i = 0
    while True:
        slot = (h + i) % size
        if table[slot] is None:
            table[slot] = key
            print(f"  키 {key:2d}: h({key}) = {key}%13 = {h}, "
                  f"슬롯 {slot}에 저장" + (f" (충돌 {i}회 후)" if i > 0 else ""))
            return
        i += 1

def print_table(table, title):
    print(f"\n{'='*40}")
    print(f" {title}")
    print(f"{'='*40}")
    print(f" 슬롯 | 값")
    print(f"------+------")
    for i, v in enumerate(table):
        print(f"  {i:2d}  |  {str(v) if v is not None else '-':>4}")
    print(f"{'='*40}")

# 크기 13의 해시 테이블 초기화
table_linear = [None] * 13
keys = [10, 20, 30, 40, 33, 46, 50, 60]

print("[선형 탐색] h'(x) = (h(x) + i) % 13, h(x) = x%13")
print("-" * 40)
for key in keys:
    linear_probing_insert(table_linear, key)

print_table(table_linear, "최종 해시 테이블 (선형 탐색)")

[선형 탐색] h'(x) = (h(x) + i) % 13, h(x) = x%13
----------------------------------------
  키 10: h(10) = 10%13 = 10, 슬롯 10에 저장
  키 20: h(20) = 20%13 = 7, 슬롯 7에 저장
  키 30: h(30) = 30%13 = 4, 슬롯 4에 저장
  키 40: h(40) = 40%13 = 1, 슬롯 1에 저장
  키 33: h(33) = 33%13 = 7, 슬롯 8에 저장 (충돌 1회 후)
  키 46: h(46) = 46%13 = 7, 슬롯 9에 저장 (충돌 2회 후)
  키 50: h(50) = 50%13 = 11, 슬롯 11에 저장
  키 60: h(60) = 60%13 = 8, 슬롯 12에 저장 (충돌 4회 후)

 최종 해시 테이블 (선형 탐색)
 슬롯 | 값
------+------
   0  |     -
   1  |    40
   2  |     -
   3  |     -
   4  |    30
   5  |     -
   6  |     -
   7  |    20
   8  |    33
   9  |    46
  10  |    10
  11  |    50
  12  |    60


**풀이 과정 설명:**

| 키 | h(x) = x%13 | 탐색 과정 | 저장 슬롯 |
|----|------------|----------|----------|
| 10 | 10 | 슬롯 10 비어있음 | **10** |
| 20 | 7  | 슬롯 7 비어있음 | **7** |
| 30 | 4  | 슬롯 4 비어있음 | **4** |
| 40 | 1  | 슬롯 1 비어있음 | **1** |
| 33 | 7  | 슬롯 7 충돌(20) → 슬롯 8 비어있음 | **8** |
| 46 | 7  | 슬롯 7 충돌(20) → 슬롯 8 충돌(33) → 슬롯 9 비어있음 | **9** |
| 50 | 11 | 슬롯 11 비어있음 | **11** |
| 60 | 8  | 슬롯 8 충돌(33) → 슬롯 9 충돌(46) → 슬롯 10 충돌(10) → 슬롯 11 충돌(50) → 슬롯 12 비어있음 | **12** |

---
## 문제 2-05. 이차 탐색(Quadratic Probing)으로 키 삽입

In [ ]:
def quadratic_probing_insert(table, key, size=13):
    """이차 탐색으로 해시 테이블에 키 삽입"""
    h = key % size
    i = 0
    while True:
        slot = (h + i * i) % size
        if table[slot] is None:
            table[slot] = key
            probes = f"충돌 {i}회 후 " if i > 0 else ""
            print(f"  키 {key:2d}: h({key}) = {h}, "
                  f"슬롯 ({h}+{i}²)%13 = {slot}에 저장 {probes}")
            return
        i += 1

# 크기 13의 해시 테이블 초기화
table_quad = [None] * 13

print("[이차 탐색] h'(x) = (h(x) + i²) % 13, h(x) = x%13")
print("-" * 45)
for key in keys:
    quadratic_probing_insert(table_quad, key)

print_table(table_quad, "최종 해시 테이블 (이차 탐색)")

**풀이 과정 설명:**

| 키 | h(x)=x%13 | i=0 | i=1 | i=2 | i=3 | i=4 | 저장 슬롯 |
|----|-----------|-----|-----|-----|-----|-----|----------|
| 10 | 10 | 슬롯 10 ✓ | | | | | **10** |
| 20 | 7  | 슬롯 7 ✓  | | | | | **7** |
| 30 | 4  | 슬롯 4 ✓  | | | | | **4** |
| 40 | 1  | 슬롯 1 ✓  | | | | | **1** |
| 33 | 7  | 슬롯 7 충돌 | (7+1)%13=8 ✓ | | | | **8** |
| 46 | 7  | 슬롯 7 충돌 | 슬롯 8 충돌 | (7+4)%13=11 ✓ | | | **11** |
| 50 | 11 | 슬롯 11 충돌 | (11+1)%13=12 ✓ | | | | **12** |
| 60 | 8  | 슬롯 8 충돌 | (8+1)%13=9 ✓ | | | | **9** |

---
## 문제 2-06. 더블 해싱(Double Hashing)으로 키 삽입

In [ ]:
def double_hashing_insert(table, key, size=13):
    """더블 해싱으로 해시 테이블에 키 삽입
    h'(x) = (h(x) + i * f(x)) % 13
    h(x) = x % 13
    f(x) = 1 + (x % 11)
    """
    h = key % size
    f = 1 + (key % 11)
    i = 0
    while True:
        slot = (h + i * f) % size
        if table[slot] is None:
            table[slot] = key
            probes = f"충돌 {i}회 후 " if i > 0 else ""
            print(f"  키 {key:2d}: h({key})={h}, f({key})={f}, "
                  f"슬롯 ({h}+{i}×{f})%13 = {slot}에 저장 {probes}")
            return
        print(f"    충돌: 슬롯 ({h}+{i}×{f})%13 = {slot} 이미 점유({table[slot]})")
        i += 1

# 크기 13의 해시 테이블 초기화
table_double = [None] * 13

print("[더블 해싱] h'(x) = (h(x) + i·f(x)) % 13")
print("           h(x) = x%13,  f(x) = 1 + (x%11)")
print("-" * 50)
for key in keys:
    double_hashing_insert(table_double, key)
    print()

print_table(table_double, "최종 해시 테이블 (더블 해싱)")

**풀이 과정 설명:**

h(x) = x%13, f(x) = 1 + (x%11)

| 키 | h(x) | f(x) | 탐색 과정 | 저장 슬롯 |
|----|------|------|----------|----------|
| 10 | 10 | 1+(10%11)=1+10=11 | 슬롯 10 비어있음 | **10** |
| 20 | 7  | 1+(20%11)=1+9=10  | 슬롯 7 비어있음 | **7** |
| 30 | 4  | 1+(30%11)=1+8=9   | 슬롯 4 비어있음 | **4** |
| 40 | 1  | 1+(40%11)=1+7=8   | 슬롯 1 비어있음 | **1** |
| 33 | 7  | 1+(33%11)=1+0=1   | 슬롯 7 충돌(20) → (7+1×1)%13=8 비어있음 | **8** |
| 46 | 7  | 1+(46%11)=1+2=3   | 슬롯 7 충돌(20) → (7+1×3)%13=10 충돌(10) → (7+2×3)%13=0 비어있음 | **0** |
| 50 | 11 | 1+(50%11)=1+6=7   | 슬롯 11 비어있음 | **11** |
| 60 | 8  | 1+(60%11)=1+5=6   | 슬롯 8 충돌(33) → (8+1×6)%13=1 충돌(40) → (8+2×6)%13=7 충돌(20) → (8+3×6)%13=0 충돌(46) → (8+4×6)%13=6 비어있음 | **6** |

---
## 문제 2-07. 체이닝에서 새 키를 리스트 맨 뒤에 삽입할 때의 효율성 변화

일반적으로 체이닝에서 새 키는 맨 앞(head)에 삽입한다. 이를 맨 뒤(tail)에 삽입하도록 변경할 경우의 효율성 변화를 분석한다.

### 비교 분석

| 연산 | 맨 앞 삽입 | 맨 뒤 삽입 |
|------|-----------|----------|
| **삽입** | O(1) — head에 바로 연결 | O(n) — tail까지 순회 후 연결 |
| **성공적 검색** | 평균 O(1 + α/2) | 평균 O(1 + α/2) (동일) |
| **실패하는 검색** | O(1 + α) | O(1 + α) (동일) |
| **삭제** | O(1 + α) | O(1 + α) (동일) |

(단, α = n/m 은 적재율)

### 결론

- **삽입 성능이 저하**된다.  
  맨 앞 삽입은 O(1)이지만, 맨 뒤 삽입은 tail 포인터가 없으면 리스트 끝까지 순회해야 하므로 O(n)이 된다.  
  (단, tail 포인터를 별도로 유지하면 O(1)로 개선 가능)
- **검색 및 삭제 성능은 변화 없다.**  
  검색은 어차피 전체 리스트를 순회해야 하므로 삽입 위치와 무관하게 동일한 성능을 보인다.
- 즉, tail 포인터 없이 맨 뒤 삽입을 구현하면 **삽입 효율만 나빠지고** 나머지는 동일하다.

---
## 문제 2-08. 체이닝에서 연결 리스트를 정렬된 형태로 유지할 때의 효율성 변화

각 버킷의 연결 리스트를 **정렬된 순서**로 유지하는 경우, 각 연산의 효율성 변화를 분석한다.

### (1) 삽입 (Insertion)

- **정렬 유지 전**: O(1) — 항상 리스트 맨 앞에 삽입
- **정렬 유지 후**: O(n) — 올바른 위치를 찾기 위해 평균적으로 리스트의 절반을 순회
- **결론**: 삽입 성능 **저하**

### (2) 삭제 (Deletion)

- **정렬 유지 전**: O(n) — 찾을 때까지 순회, 못 찾으면 끝까지 순회
- **정렬 유지 후**: O(n) — 찾을 때까지 순회하지만, 찾는 키보다 큰 값을 만나면 조기 종료 가능
- **결론**: 평균적으로 탐색 비용 **소폭 감소** (조기 종료 효과), 그러나 worst case는 동일

### (3) 성공적인 검색 (Successful Search)

- **정렬 유지 전**: 평균 리스트 길이의 절반 탐색 → O(1 + α/2)
- **정렬 유지 후**: 동일하게 평균 리스트 길이의 절반 탐색 → O(1 + α/2)
- **결론**: 성능 **변화 없음** (찾는 키가 어느 위치에 있는지는 정렬 여부와 무관하게 평균적으로 동일)

### (4) 실패하는 검색 (Unsuccessful Search)

- **정렬 유지 전**: 리스트 끝까지 모두 탐색 → O(1 + α)
- **정렬 유지 후**: 찾는 키보다 큰 값을 만나는 순간 **조기 종료** 가능 → 평균 O(1 + α/2)
- **결론**: 실패하는 검색 성능이 **크게 향상** (절반의 탐색으로 실패 판정 가능)

### 요약 비교표

| 연산 | 정렬 유지 전 | 정렬 유지 후 | 변화 |
|------|------------|------------|------|
| 삽입 | O(1) | O(α/2) |  저하 |
| 삭제 | O(α) | O(α/2) 평균 |  소폭 향상 |
| 성공적 검색 | O(1 + α/2) | O(1 + α/2) |  동일 |
| 실패하는 검색 | O(1 + α) | O(1 + α/2) |  향상 |

(α = n/m, 적재율)

**결론**: 정렬 유지는 삽입 비용을 높이지만, **실패하는 검색**에서 뚜렷한 이점을 제공한다.  
검색(특히 실패 검색)이 빈번한 환경에서는 정렬 유지가 유리하며, 삽입이 빈번한 환경에서는 불리하다.